In [2]:
import pandas as pd

df = pd.read_csv("data/events.csv")
df["dt"] = pd.to_datetime(df["timestamp"], unit="ms")

P1 = ("2015-06-01", "2015-06-14")   # training
P2 = ("2015-06-15", "2015-06-21")   # testing

def filter_period(d, start, end):
    s = pd.Timestamp(start)
    e = pd.Timestamp(end) + pd.Timedelta(days=1)
    return d[(d["dt"] >= s) & (d["dt"] < e)]

train = filter_period(df, *P1)
test  = filter_period(df, *P2)

print("Train:", len(train), "event |", train.visitorid.nunique(), "visitor |", train.itemid.nunique(), "item")
print("Test :", len(test),  "event |", test.visitorid.nunique(),  "visitor |", test.itemid.nunique(),  "item")
print(train.event.value_counts())

Train: 278297 event | 148909 visitor | 76420 item
Test : 145976 event | 76996 visitor | 50854 item
event
view           269671
addtocart        6486
transaction      2140
Name: count, dtype: int64


In [3]:
train.drop(columns="dt").to_csv("data/events_train.csv", index=False)
test.drop(columns="dt").to_csv("data/events_test.csv", index=False)
print("Tersimpan")

Tersimpan


In [7]:
def import_batch(tx, batch):
    cypher_query = """
    UNWIND $batch AS row
    MERGE (v:Visitor {visitorId: row.visitorid})
    MERGE (i:Item {itemId: row.itemid})

    WITH v, i, row
    CALL (v, i, row) {
        WITH v, i, row WHERE row.event = 'view'
        CREATE (v)-[:VIEWED {timestamp: row.timestamp}]->(i)

        UNION ALL

        WITH v, i, row WHERE row.event = 'addtocart'
        CREATE (v)-[:ADDED_TO_CART {timestamp: row.timestamp}]->(i)

        UNION ALL

        WITH v, i, row WHERE row.event = 'transaction'
        CREATE (v)-[:BOUGHT {timestamp: row.timestamp, transactionId: row.transactionid}]->(i)
    }
    """
    tx.run(cypher_query, batch=batch)

In [8]:
import pandas as pd

driver.execute_query("MATCH (n) DETACH DELETE n", database_=DB)

csv_path = "data/events_train.csv"
batch_size = 5000
total_rows = len(train)
total_chunks = (total_rows // batch_size) + 1

with driver.session(database=DB) as session:
    create_constraints(session)
    chunks = pd.read_csv(csv_path, chunksize=batch_size)
    for chunk in tqdm(chunks, total=total_chunks, desc="Mengimpor Batch Data"):
        chunk["transactionid"] = chunk["transactionid"].fillna("").astype(str)
        records = chunk.to_dict(orient="records")
        session.execute_write(import_batch, records)

print("Proses impor berhasil")

Mengimpor Batch Data:   0%|          | 0/56 [00:00<?, ?it/s]

Proses impor berhasil


In [9]:
r, _, _ = driver.execute_query(
    "MATCH (n) RETURN labels(n) AS label, count(*) AS jumlah", database_=DB)
for x in r: print(x["label"], x["jumlah"])

r, _, _ = driver.execute_query(
    "MATCH ()-[r]->() RETURN type(r) AS tipe, count(*) AS jumlah", database_=DB)
for x in r: print(x["tipe"], x["jumlah"])

['Visitor'] 148909
['Item'] 76420
VIEWED 269671
ADDED_TO_CART 6486
BOUGHT 2140
